# 01 - PA2 Data Preparation

This notebook defines the data preparation strategy for Practical Application 2:
probabilistic supervised classification.

The same IBM HR Analytics Employee Attrition dataset used in Practical
Application 1 is retained.

The objective is to identify the different predictor types and document the
representations required by the probabilistic classifiers studied in the
course.

No classification model is trained in this notebook.

In [24]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

DATA_PATH = "../../data/raw/WA_Fn-UseC_-HR-Employee-Attrition.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)

Dataset shape: (1470, 35)


## 1. Removal of Non-Predictive Variables

The same non-predictive attributes identified in Practical Application 1 are
removed:

- `EmployeeNumber`, because it is an identifier.
- `EmployeeCount`, because it is constant.
- `Over18`, because it is constant.
- `StandardHours`, because it is constant.

These removals are not considered feature selection. They simply exclude
attributes that cannot provide useful discriminatory information.

In [25]:
columns_to_remove = [
    "EmployeeNumber",
    "EmployeeCount",
    "Over18",
    "StandardHours"
]

df_model = df.drop(columns=columns_to_remove)

print("Original shape:", df.shape)
print("Model dataset shape:", df_model.shape)

Original shape: (1470, 35)
Model dataset shape: (1470, 31)


## 2. Predictors and Target

`Attrition` is the binary target variable.

The target is encoded as:

- `No` -> 0
- `Yes` -> 1

Therefore, the positive class corresponds to employee attrition.

In [26]:
X = df_model.drop(columns=["Attrition"])

y = df_model["Attrition"].map({
    "No": 0,
    "Yes": 1
})

print("Predictor matrix shape:", X.shape)
print("Target shape:", y.shape)

target_summary = pd.DataFrame({
    "Count": y.value_counts(),
    "Percentage": (
        y.value_counts(normalize=True)
        .mul(100)
        .round(2)
    )
})

display(target_summary)

Predictor matrix shape: (1470, 30)
Target shape: (1470,)


,Count,Percentage
Attrition,,
0,1233,83.88
1,237,16.12


## 3. Predictor Types

The dataset contains numerical, ordinal, and categorical predictors.

This distinction is important in probabilistic classification because different
models make different assumptions about the distribution and representation of
the input variables.

In [27]:
numerical_features = (
    X.select_dtypes(include=np.number)
    .columns
    .tolist()
)

categorical_features = (
    X.select_dtypes(exclude=np.number)
    .columns
    .tolist()
)

print(
    f"Numerical predictors: {len(numerical_features)}"
)

print(numerical_features)

print(
    f"\nCategorical predictors: {len(categorical_features)}"
)

print(categorical_features)

Numerical predictors: 23
['Age', 'DailyRate', 'DistanceFromHome', 'Education', 'EnvironmentSatisfaction', 'HourlyRate', 'JobInvolvement', 'JobLevel', 'JobSatisfaction', 'MonthlyIncome', 'MonthlyRate', 'NumCompaniesWorked', 'PercentSalaryHike', 'PerformanceRating', 'RelationshipSatisfaction', 'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance', 'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager']

Categorical predictors: 7
['BusinessTravel', 'Department', 'EducationField', 'Gender', 'JobRole', 'MaritalStatus', 'OverTime']


## 4. Ordinal Predictors

Several integer-valued attributes represent ordered categories rather than
arbitrary continuous measurements.

These variables preserve their original ordinal coding because the ordering is
meaningful.

In [28]:
ordinal_features = [
    "Education",
    "EnvironmentSatisfaction",
    "JobInvolvement",
    "JobLevel",
    "JobSatisfaction",
    "PerformanceRating",
    "RelationshipSatisfaction",
    "StockOptionLevel",
    "WorkLifeBalance"
]

continuous_features = [
    feature
    for feature in numerical_features
    if feature not in ordinal_features
]

print(
    f"Ordinal predictors: {len(ordinal_features)}"
)

print(ordinal_features)

print(
    f"\nRemaining numerical predictors: "
    f"{len(continuous_features)}"
)

print(continuous_features)

Ordinal predictors: 9
['Education', 'EnvironmentSatisfaction', 'JobInvolvement', 'JobLevel', 'JobSatisfaction', 'PerformanceRating', 'RelationshipSatisfaction', 'StockOptionLevel', 'WorkLifeBalance']

Remaining numerical predictors: 14
['Age', 'DailyRate', 'DistanceFromHome', 'HourlyRate', 'MonthlyIncome', 'MonthlyRate', 'NumCompaniesWorked', 'PercentSalaryHike', 'TotalWorkingYears', 'TrainingTimesLastYear', 'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager']


## 5. Categorical Predictor Values

The possible values of each categorical predictor are inspected because
probabilistic classifiers may require explicit categorical probability
estimation or numerical encoding depending on the model.

In [29]:
for col in categorical_features:

    print(f"\n--- {col} ---")

    values = (
        X[col]
        .dropna()
        .unique()
        .tolist()
    )

    print(sorted(values))


--- BusinessTravel ---
['Non-Travel', 'Travel_Frequently', 'Travel_Rarely']

--- Department ---
['Human Resources', 'Research & Development', 'Sales']

--- EducationField ---
['Human Resources', 'Life Sciences', 'Marketing', 'Medical', 'Other', 'Technical Degree']

--- Gender ---
['Female', 'Male']

--- JobRole ---
['Healthcare Representative', 'Human Resources', 'Laboratory Technician', 'Manager', 'Manufacturing Director', 'Research Director', 'Research Scientist', 'Sales Executive', 'Sales Representative']

--- MaritalStatus ---
['Divorced', 'Married', 'Single']

--- OverTime ---
['No', 'Yes']


## 6. Missing Values

Missing values are checked because probabilistic parameter estimation would
require an explicit treatment if incomplete observations were present.

In [30]:
missing_values = X.isnull().sum()

missing_variables = (
    missing_values[missing_values > 0]
    .sort_values(ascending=False)
)

if missing_variables.empty:
    print("No missing values were found.")
else:
    display(missing_variables)

No missing values were found.


## 7. Model-Specific Data Representations

A single transformed feature matrix is not necessarily appropriate for every
probabilistic classifier.

Different representations will therefore be used according to the model family.

### Mixed Naive Bayes

The course explicitly presents a Naive Bayes classifier with discrete and
continuous predictors.

Continuous predictors will be modeled through continuous class-conditional
distributions, while categorical predictors will be represented through
class-conditional category probabilities.

### Logistic Regression

Categorical predictors require numerical encoding. One-hot encoding will be
used for nominal predictors, while ordinal predictors retain their ordered
numerical representation.

Continuous predictors may be standardized inside the model pipeline.

### Linear Discriminant Analysis

LDA requires a numerical representation. Nominal categorical predictors will
therefore be encoded numerically, while continuous and ordinal predictors will
remain numerical.

The exact encoding strategy will be defined inside the model pipeline and
evaluated without using validation information.

### Quadratic Discriminant Analysis

QDA also requires a numerical representation and estimates a separate
covariance matrix for each target class.

The same principle of training-only preprocessing will be maintained, while
the exact representation will be selected to avoid unnecessary redundancy or
numerical instability.

In [31]:
feature_type_summary = pd.DataFrame({
    "Feature": X.columns
})

feature_type_summary["Type"] = "Continuous"

feature_type_summary.loc[
    feature_type_summary["Feature"].isin(ordinal_features),
    "Type"
] = "Ordinal"

feature_type_summary.loc[
    feature_type_summary["Feature"].isin(categorical_features),
    "Type"
] = "Categorical"

In [32]:
display(
    feature_type_summary.sort_values(
        ["Type", "Feature"]
    )
)

,Feature,Type
1,BusinessTravel,Categorical
3,Department,Categorical
6,EducationField,Categorical
8,Gender,Categorical
12,JobRole,Categorical
14,MaritalStatus,Categorical
18,OverTime,Categorical
0,Age,Continuous
2,DailyRate,Continuous
4,DistanceFromHome,Continuous


In [33]:
type_counts = (
    feature_type_summary["Type"]
    .value_counts()
    .rename_axis("Predictor Type")
    .reset_index(name="Count")
)

display(type_counts)

,Predictor Type,Count
0,Continuous,14
1,Ordinal,9
2,Categorical,7


## 8. Class Distribution and Honest Evaluation

The target distribution is imbalanced, with substantially fewer attrition
cases than non-attrition cases.

The course recommends stratified k-fold cross-validation for unbalanced
datasets.

Therefore, the probabilistic classifiers will later be evaluated using
stratified cross-validation so that each fold approximately preserves the
original class distribution.

The evaluation will include both class-label metrics and probability-aware
metrics.

In [34]:
display(
    target_summary
)

,Count,Percentage
Attrition,,
0,1233,83.88
1,237,16.12


## 9. Evaluation Metrics for PA2

The probabilistic classifiers will be evaluated using metrics explicitly
covered in the course.

Classification metrics:

- Accuracy
- Sensitivity / Recall
- Specificity
- Positive Predictive Value / Precision
- F1-measure

Probability-aware evaluation:

- ROC curve
- Area Under the ROC Curve (AUC)
- Brier score

The Brier score will be particularly relevant in PA2 because it evaluates the
quality of predicted class probabilities rather than only the final predicted
class.

Balanced accuracy will not be used as a primary course metric in PA2 because
it was not part of the supplied performance-evaluation material.

## 10. Feature Selection Plan

Practical Application 2 requires the same four analyses used in Practical
Application 1:

1. All valid predictors.
2. Univariate filter feature selection.
3. Multivariate filter feature selection.
4. Wrapper feature selection.

The feature-selection methods used in PA2 will be selected from the methods
explicitly studied in the course.

The exact selectors will be implemented in later notebooks so that feature
selection remains part of the honest cross-validation procedure.

## 11. Probabilistic Classifier Families

Practical Application 2 will use representative probabilistic classifier
families studied in the course.

The planned classifier families are:

- Naive Bayes for mixed discrete and continuous predictors.
- Logistic Regression.
- Linear Discriminant Analysis (LDA).
- Quadratic Discriminant Analysis (QDA).

Only one representative implementation per classifier family will be used.
The objective is to cover the probabilistic classifier families taught in the
course, rather than every individual variant mentioned in the lecture
material.

## 12. Metaclassifier Plan

Practical Application 2 also requires the use of metaclassifiers.

Representative metaclassifier strategies studied in the course will be used
later in the practical application.

The exact set of metaclassifiers will be selected so that different ensemble
principles are represented without exhaustively implementing every variant
mentioned in the lecture material.

## 13. Data Preparation Summary

The PA2 dataset contains:

- 1470 observations.
- 30 valid original predictors.
- A binary target (`Attrition`).
- Continuous, ordinal, and categorical predictors.
- No missing values.
- An imbalanced target distribution.

Unlike PA1, PA2 will not force every classifier to use one identical
preprocessed feature matrix.

Instead, each probabilistic classifier will use a representation compatible
with its assumptions, while all preprocessing and feature-selection operations
will remain inside the honest cross-validation procedure.